In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, concat_ws, split,to_date,try_to_date
from pyspark.sql import functions as F

In [0]:
df = spark.table("novacart_catalog.001_bronze.orders")

df.display()

In [0]:
df = df.withColumn(
    "customer_id",
    F.when(
        F.trim(F.col("customer_id")).isin("NULL", "\\N", "?", "", None) | F.col("customer_id").isNull(),
        F.lit("unknown")
    ).otherwise(F.trim(F.col("customer_id")))
)
df.display()

In [0]:
df = df.withColumn(
    "new_order_status",
    F.when(F.col("order_status").isin("completed", "已完成", "completado", "abgeschlossen", "पूर्ण"), "completed")
     .when(F.col("order_status").isin("pending", "待处理", "pendiente", "ausstehend","लंबित"), "pending")
     .when(F.col("order_status").isin("shipped", "भेज दिया", "enviado", "versandt","已发货"), "shipped")
     .when(F.col("order_status").isin("cancelled", "cancelado", "storniert", "रद्द","已取消"), "cancelled")
     .when(F.trim(F.col("order_status")).isin("NULL", "\\N", "?", ""), "unknown")
     .otherwise(F.col("order_status"))
)
df.display()

In [0]:
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.expr("try_to_date(order_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(order_date, 'd/M/yyyy')"),
        F.expr("try_to_date(order_date, 'M/d/yyyy')"),
        F.expr("try_to_date(order_date, 'MM-dd-yyyy')"),
       F.col("order_date")
    )
)
display(df.limit(200))

In [0]:
df=df.withColumn("country_code", trim(col("country_code"))) \
    .withColumn("channel", trim(col("channel"))) \
    .withColumn("currency", trim(col("currency")))
df.display()

In [0]:
valid_customer_ids = [row.customer_id for row in spark.table("novacart_catalog.001_bronze.customers").select("customer_id").distinct().collect()]

df = df.withColumn(
    "is_valid_customer",
    when(col("customer_id") == "unknown", False)
    .when(col("customer_id").isin(valid_customer_ids), True)
    .otherwise(False)
).select(df.columns + ["is_valid_customer"])

df.display()

In [0]:
for col_name in df.columns:
    if len(col_name) > 0:
       new_col_name = col_name[0].upper() + col_name[1:]
    else:
       new_col_name = col_name
    df = df.withColumnRenamed(col_name, new_col_name)

df.printSchema()

In [0]:
display(df)

In [0]:
cols_to_select = [col for col in df.columns if col.lower() != "order_status" and col.lower() != "new_order_status"]
df = df.select(*cols_to_select, F.col("New_order_status").alias("Order_status"))
display(df)

In [0]:
df.write.format("delta") .mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("novacart_catalog.002_silver.orders")